In [ ]:
import os
import subprocess
import zipfile

# Download destination — change this if you want a different drive
DATA_DIR = "/Volumes/RICK/convokit/subreddit-OutOfTheLoop"

CORPUS_NAME = ""
ZIP_URL = "https://zissou.infosci.cornell.edu/convokit/datasets/subreddit-corpus/corpus-zipped/OpenLetter~-~OutlandishAlcoholics/OutOfTheLoop.corpus.zip"

zip_path  = os.path.join(DATA_DIR, f"{CORPUS_NAME}.zip")
corpus_dir = os.path.join(DATA_DIR, CORPUS_NAME)
downloads_dir = os.path.join(DATA_DIR, "downloads")
downloaded_txt = os.path.join(downloads_dir, "downloaded.txt")

os.makedirs(downloads_dir, exist_ok=True)
if not os.path.isfile(downloaded_txt):
    open(downloaded_txt, "w").close()

if not os.path.isdir(corpus_dir):
    print(f"Downloading {CORPUS_NAME} (~84 GB) to {zip_path}")
    print("Progress is shown below. Safe to interrupt — curl -C - will resume where it left off.\n")
    subprocess.run(
        ["curl", "-C", "-", "-L", "--progress-bar", "-o", zip_path, ZIP_URL],
        check=True,
    )
    print("\nExtracting zip...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_DIR)
    # Register with convokit so it finds the corpus without re-downloading
    with open(downloaded_txt, "a") as f:
        f.write(f"{CORPUS_NAME}$#${DATA_DIR}$#$0\n")
    print("Done.")
else:
    print(f"Corpus already at {corpus_dir}")

Corpus already at /Volumes/RICK/convokit/subreddit-OutOfTheLoop/


In [ ]:
import json


def stream_content():
    f = open(os.path.join(corpus_dir, "utterances.jsonl"))
    for jsonline in f:
        yield json.loads(jsonline)

In [ ]:
import os

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [4]:
i = 0
n = 0
MAX_SIZE=100000

text_timestamp = []

schema = pa.schema([
    ("timestamp", pa.int64()),
    ("text", pa.large_string())
])

writer = pq.ParquetWriter("/Volumes/RICK/convokit/OutOfTheLoop.parquet", schema)

for utterance in stream_content():
    i+=1
    if i <=MAX_SIZE:
        text_timestamp.append((utterance["timestamp"], utterance["text"]))
    else:
        n+=1
        # write to the parquet file
        text_timestamp_df = pd.DataFrame(text_timestamp, columns=["timestamp","text"])
        batch = pa.RecordBatch.from_pandas(text_timestamp_df)
        writer.write_batch(batch)
        text_timestamp = []
        i=0
writer.close()

## Temporal Referencing with Skip-gram Negative Sampling Embeddings


The original paper uses the codecs package instead of the typical file package to read in the text

In [ ]:
import re
from datetime import datetime, timezone

import pyarrow.parquet as pq

SENT_SPLIT = re.compile(r"[.!?\n]+")
TOKEN = re.compile(r"[a-z]+(?:'[a-z]+)?")

class SentenceStream:
    def __init__(self, path, batch_size=10_000,
                 text_col="text", time_col="timestamp"):
        self.path = path
        self.batch_size = batch_size
        self.text_col = text_col
        self.time_col = time_col

    def __iter__(self):
        pf = pq.ParquetFile(path)
        for batch in pf.iter_batches(batch_size=self.batch_size,
                                          columns=[self.time_col, self.text_col]):
            for ts, text in zip(batch[self.time_col], batch[self.text_col]):
                ts, text = ts.as_py(), text.as_py()
                if ts is None or not text:
                    continue
                year = datetime.fromtimestamp(ts, timezone.utc).year
                for chunk in SENT_SPLIT.split(text):
                    chunk = chunk.strip()
                    if not chunk:
                        continue
                    toks = TOKEN.findall(chunk.lower())
                    if toks:
                        yield year, toks

In [26]:
corpus = pq.ParquetFile("/Volumes/RICK/convokit/OutOfTheLoop.parquet")

path = "/Volumes/RICK/convokit/OutOfTheLoop.parquet"

corpus_stream = SentenceStream(path)

print(next(iter(corpus_stream)))


(2016, ['i', 'keep', 'hearing', 'about', 'him', 'around', 'reddit', 'lately'])


In [ ]:
from collections import defaultdict

corpus = pq.ParquetFile("/Volumes/RICK/convokit/OutOfTheLoop.parquet")

word_freqs = {
    str(year): defaultdict(int)
    for year in range(2005, 2019)
}

for sentence in SentenceStream(corpus):
    year = str(sentence[0])

    for word in sentence[1]:
        word_freqs[year][word] += 1

In [29]:
from collections import Counter

print(Counter(word_freqs["2015"]).most_common(20))

[('the', 565313), ('to', 350898), ('a', 336283), ('and', 278791), ('of', 264040), ('i', 220890), ('it', 211002), ('is', 204478), ('that', 204110), ('in', 174566), ('you', 146332), ('for', 135069), ('this', 104769), ('was', 104107), ('but', 98269), ('on', 98182), ('are', 90457), ('they', 88029), ('not', 87308), ('with', 79080)]


In [30]:
from collections import Counter, defaultdict

PATH = "/Volumes/RICK/convokit/OutOfTheLoop.parquet"

total, by_year = Counter(), defaultdict(Counter)
for year, toks in SentenceStream(PATH):
    total.update(toks)
    by_year[year].update(toks)

MIN_COUNT = 10                      # toy run; 100 for real corpora
vocab = {w for w, c in total.items() if c >= MIN_COUNT}

In [ ]:
import logging
import time

logging.basicConfig(format="%(asctime)s | %(message)s", level=logging.INFO)

WINDOW = 5
OUT = "/Volumes/RICK/convokit/outofloop_pairs.txt"
date2ref = {y: str(y) for y in range(2005, 2019)}   # swap ranges in here
# later for merged bins
start, n_sent, n_pairs = time.time(), 0, 0

with open(OUT, "w", encoding="utf-8") as f:
    for year, sent in SentenceStream(PATH):
        ref = date2ref.get(year)
        if ref is None:
            continue
        n_sent += 1
        L = len(sent)
        for i, w in enumerate(sent):
            if w not in vocab:                      # gate 1: target in vocab
                continue
            center = f"{w}_{ref}"
            for j in range(max(i - WINDOW, 0), min(i + WINDOW + 1, L)):
                if j == i:
                    continue
                c = sent[j]
                if c in vocab:                      # gate 2: context in vocab
                    f.write(f"{center} {c}\n")
                    n_pairs += 1
        if n_sent % 100_000 == 0:
            rate = n_sent / (time.time() - start)
            logging.info(f"{n_sent:,} sents | {n_pairs:,} pairs | {rate:,.0f} sent/s")

2026-06-11 16:00:52,274 | INFO | 100,000 sents | 5,439,838 pairs | 45,420 sent/s
2026-06-11 16:00:53,941 | INFO | 200,000 sents | 9,571,818 pairs | 51,700 sent/s
2026-06-11 16:00:55,681 | INFO | 300,000 sents | 14,199,296 pairs | 53,487 sent/s
2026-06-11 16:00:57,914 | INFO | 400,000 sents | 19,008,926 pairs | 51,017 sent/s
2026-06-11 16:01:00,151 | INFO | 500,000 sents | 24,218,882 pairs | 49,613 sent/s
2026-06-11 16:01:03,260 | INFO | 600,000 sents | 32,376,918 pairs | 45,499 sent/s
2026-06-11 16:01:06,454 | INFO | 700,000 sents | 40,714,324 pairs | 42,733 sent/s
2026-06-11 16:01:10,317 | INFO | 800,000 sents | 49,157,088 pairs | 39,518 sent/s
2026-06-11 16:01:13,760 | INFO | 900,000 sents | 57,680,844 pairs | 37,995 sent/s
2026-06-11 16:01:16,973 | INFO | 1,000,000 sents | 66,043,620 pairs | 37,174 sent/s
2026-06-11 16:01:20,586 | INFO | 1,100,000 sents | 74,578,556 pairs | 36,050 sent/s
2026-06-11 16:01:24,021 | INFO | 1,200,000 sents | 83,450,314 pairs | 35,348 sent/s
2026-06-11 1